# Second Ear — train the risk model

Scaffold notebook. Every cell below this one is written in Phase 2, on Colab, with a GPU
runtime. Read `ml/README.md` first; the dataset facts there were checked against the files.

**Required output**

- `models/risk-scorer/` — DistilBERT weights + tokenizer, `config.json` at the root. The server
  picks it up automatically (`server/scorer.py` checks for `config.json`).
- `eval/results/model.json` — precision, recall, F1 with n, on full test dialogues and on
  40 % prefixes, plus training wall-clock seconds.

**Order of operations (do not reorder; each step prevents a leak or a mismatch)**

1. Load `BothBosu/scam-dialogue` (train/test CSVs). Print columns and 3 rows before anything else.
2. Strip inline `caller:` / `receiver:` tags. Live transcripts have no speaker labels.
3. Split by conversation into train/val/test **before** augmentation; the given test file is test.
4. Augment train and val only: prefixes at 20/40/60/80/100 % of each dialogue's words, same label.
5. Fine-tune `distilbert-base-uncased`: `max_length=256`, `truncation_side="left"`, 3 epochs,
   `lr=2e-5`, fp16. Checkpoint to Drive every epoch.
6. Evaluate: precision, recall, F1 on full test and on 40 % prefixes. Write `model.json`.
7. Save the model and tokenizer; zip the folder for download into `models/risk-scorer/`.
